# Kémzy LiveAvatar — Run All

This notebook prepares the pinned PersonaLive GPU runtime, reuses mounted weights when available, and connects the worker outbound to the Kémzy Render gateway. Add Kaggle secret `GPU_WORKER_SECRET` (must match Render). Optional secret `KEMZY_RENDER_WS_URL` defaults to `https://kemzy-api-production.onrender.com`. Enable **GPU** and **Internet**, then choose **Run All** and leave the notebook running. Kaggle sessions are temporary; this notebook does not guarantee continuous 24/7 service.

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib, pathlib, types
from pathlib import Path

print("=== KÉMZY LIVE STREAM: RUN ALL ===", flush=True)
assert Path("/kaggle/working").exists(), "Open this notebook in Kaggle."
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU in Notebook Settings."
print("GPU:", torch.cuda.get_device_name(0), flush=True)

# Read secrets without ever printing them.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GPU_WORKER_SECRET = secrets.get_secret("GPU_WORKER_SECRET")
    try:
        KEMZY_RENDER_WS_URL = secrets.get_secret("KEMZY_RENDER_WS_URL")
    except Exception:
        KEMZY_RENDER_WS_URL = None
except Exception as exc:
    raise RuntimeError("Add GPU_WORKER_SECRET in Kaggle > Add-ons > Secrets, then rerun.") from exc
if not GPU_WORKER_SECRET:
    raise RuntimeError("GPU_WORKER_SECRET is missing in Kaggle Secrets.")
KEMZY_RENDER_WS_URL = (KEMZY_RENDER_WS_URL or "https://kemzy-api-production.onrender.com").strip()
os.environ["GPU_WORKER_SECRET"] = GPU_WORKER_SECRET
os.environ["KEMZY_RENDER_WS_URL"] = KEMZY_RENDER_WS_URL
os.environ["ACCELERATION"] = "none"
print("Render gateway:", KEMZY_RENDER_WS_URL, flush=True)

# Get the current Kémzy integration code.
KEMZY = Path("/kaggle/working/Kemzy-LiveAvatar")
if KEMZY.exists():
    subprocess.run(["git", "-C", str(KEMZY), "fetch", "origin", "feature/backend-render-gateway"], check=True)
    subprocess.run(["git", "-C", str(KEMZY), "reset", "--hard", "origin/feature/backend-render-gateway"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "feature/backend-render-gateway",
                    "https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git", str(KEMZY)], check=True)

# Use the exact PersonaLive revision already integrated with Kémzy.
ROOT = Path("/kaggle/working/PersonaLive")
PIN = "abdd112e01dcf7d89122c2e5efa29fcff0669740"
if ROOT.exists():
    head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
    if head.returncode != 0 or head.stdout.strip() != PIN:
        shutil.rmtree(ROOT)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/GVCLab/PersonaLive.git", str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", PIN], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", PIN], check=True)
assert subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip() == PIN
print("PersonaLive pinned revision:", PIN, flush=True)

# Keep Kaggle's Torch/CUDA and protobuf environment intact.
if "tensorflow.tools.docs" not in sys.modules:
    tf = types.ModuleType("tensorflow")
    tf_tools = types.ModuleType("tensorflow.tools")
    tf_docs = types.ModuleType("tensorflow.tools.docs")
    tf_docs.doc_controls = types.SimpleNamespace()
    tf_tools.docs = tf_docs
    tf.tools = tf_tools
    sys.modules["tensorflow"] = tf
    sys.modules["tensorflow.tools"] = tf_tools
    sys.modules["tensorflow.tools.docs"] = tf_docs

def module_version(module_name):
    try:
        module = importlib.import_module(module_name)
        return str(getattr(module, "__version__", "unknown"))
    except Exception:
        return None

packages = [
    ("mediapipe", "mediapipe==0.10.13"),
    ("av", "av==18.1.0"),
    ("decord", "decord==0.6.0"),
    ("websocket", "websocket-client==1.8.0"),
]
to_install = []
for module_name, package_spec in packages:
    version = module_version(module_name)
    print(module_name, version or "not installed", flush=True)
    if version is None or (module_name != "websocket" and version != package_spec.split("==")[1]):
        to_install.append(package_spec)
if to_install:
    print("Installing missing runtime packages without changing Torch/protobuf:", to_install, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *to_install], check=True)
import diffusers
print("Torch:", torch.__version__, "| Diffusers:", diffusers.__version__, "| CUDA:", torch.cuda.is_available(), flush=True)

# Reuse exact-size weights already mounted in Kaggle. Download only if needed.
PR = ROOT / "pretrained_weights"
PR.mkdir(parents=True, exist_ok=True)
EXPECTED_SIZES = {
 "personalive/denoising_unet.pth":4927015578,
 "personalive/motion_encoder.pth":246719031,
 "personalive/motion_extractor.pth":112545505,
 "personalive/pose_guider.pth":4351700,
 "personalive/reference_unet.pth":3438324340,
 "personalive/temporal_module.pth":1817903019,
 "sd-vae-ft-mse/diffusion_pytorch_model.bin":334707217,
 "sd-vae-ft-mse/config.json":547,
 "sd-image-variations-diffusers/image_encoder/pytorch_model.bin":1215993867,
 "sd-image-variations-diffusers/image_encoder/config.json":703,
 "sd-image-variations-diffusers/unet/diffusion_pytorch_model.bin":3438350225,
 "sd-image-variations-diffusers/unet/config.json":471,
 "sd-image-variations-diffusers/model_index.json":545,
}
def valid_weight(path, rel):
    return path.is_file() and path.stat().st_size == EXPECTED_SIZES[rel]
def find_weight(rel):
    name = Path(rel).name
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        for found in base.rglob(name):
            try:
                if found.resolve() == (PR / rel).resolve():
                    continue
                if valid_weight(found, rel):
                    return found
            except (OSError, RuntimeError):
                pass
    return None

missing = []
for rel in EXPECTED_SIZES:
    dest = PR / rel
    if valid_weight(dest, rel):
        continue
    source = find_weight(rel)
    if source:
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            dest.unlink()
        dest.symlink_to(source)
        print("Reusing mounted weight:", rel, flush=True)
    else:
        if dest.exists() or dest.is_symlink():
            dest.unlink()
        missing.append(rel)

if missing:
    print("Downloading missing model files inside Kaggle. This can take time and several GB.", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "huggingface_hub==0.25.1"], check=True)
    subprocess.run([sys.executable, "tools/download_weights.py"], cwd=str(ROOT), check=True)
    invalid = [rel for rel, size in EXPECTED_SIZES.items() if not valid_weight(PR / rel, rel)]
    if invalid:
        raise RuntimeError("Weights still missing or size-invalid after download: " + ", ".join(invalid))
print("All 13 PersonaLive weight/config files verified.", flush=True)

# Apply the repository's pinned compatibility patch.
patch = KEMZY / "tools" / "personalive_compat_patch.py"
subprocess.run([sys.executable, str(patch), str(ROOT)], check=True)
os.environ["PERSONALIVE_ROOT"] = str(ROOT)
os.environ["PERSONALIVE_CONFIG"] = "./configs/prompts/personalive_online.yaml"
os.environ["PYTHONPATH"] = str(ROOT)
worker = KEMZY / "backend" / "kaggle_worker" / "personalive_outbound_worker.py"
assert worker.is_file(), f"Worker missing: {worker}"
print("Setup complete. Starting persistent Kémzy GPU worker. Leave this cell running.", flush=True)
os.execv(sys.executable, [sys.executable, str(worker)])
